<a href="https://colab.research.google.com/github/Nour-Tamimi/BinXtraining/blob/main/Week7/Day4/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch

In [2]:
# Load a pre-trained transformer with the Hugging Face pipeline and run it on sample text
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

result = classifier("This internship is excellent!")
print(result)

result2 = classifier("I am really disappointed with this service.")
print(result2)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998610019683838}]
[{'label': 'NEGATIVE', 'score': 0.999702513217926}]


In [15]:
!pip install datasets

from datasets import load_dataset

# Load AG News
dataset = load_dataset("fancyzhx/ag_news")

# Shape
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [16]:
from transformers import pipeline

label_names = ["World", "Sports", "Business", "Sci/Tech"]

# Load a zero-shot classification model (we specify the model explicitly)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Take a small sample first: 20 examples
test_sample = dataset["test"].select(range(20))

# Try it on a single example first
sample_text = test_sample[0]["text"]
true_label = label_names[test_sample[0]["label"]]

result = classifier(sample_text, candidate_labels=label_names)

print("Text:", sample_text)
print("True label:", true_label)
print("Predicted label:", result["labels"][0])
print("Confidence score:", result["scores"][0])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Text: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
True label: Business
Predicted label: Business
Confidence score: 0.5836386680603027


In [17]:
import time

test_sample = dataset["test"].select(range(200))

correct = 0
start_time = time.time()

for example in test_sample:
    result = classifier(example["text"], candidate_labels=label_names)
    predicted_label = label_names.index(result["labels"][0])
    if predicted_label == example["label"]:
        correct += 1

accuracy = correct / len(test_sample)
elapsed_time = time.time() - start_time

print(f"Transformer (zero-shot) Accuracy: {accuracy:.2%}")
print(f"Time taken: {elapsed_time:.1f} seconds")

Transformer (zero-shot) Accuracy: 70.50%
Time taken: 30.0 seconds


In [18]:
# Shuffle the full training set first, THEN take our sample
train_sample = dataset["train"].shuffle(seed=42).select(range(5000))
test_sample_lstm = dataset["test"].shuffle(seed=42).select(range(200))

train_texts = train_sample["text"]
train_labels = train_sample["label"]

test_texts = test_sample_lstm["text"]
test_labels = test_sample_lstm["label"]

# Re-tokenize with the shuffled data
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

train_padded = pad_sequences(train_sequences, maxlen=max_length, padding="post", truncating="post")
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding="post", truncating="post")

train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

print("Label distribution in train sample:", np.bincount(train_labels))

Label distribution in train sample: [1253 1273 1150 1324]


In [20]:
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Input(shape=(max_length,)),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),
    LSTM(64),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Stop training when val_loss stops improving, and restore the best weights
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,              # wait 2 extra epochs before giving up, in case it's a temporary blip
    restore_best_weights=True
)

history = model.fit(
    train_padded, train_labels,
    epochs=30,               # set it high — EarlyStopping will cut it short automatically
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.5722 - loss: 0.9895 - val_accuracy: 0.8440 - val_loss: 0.4503
Epoch 2/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8940 - loss: 0.3430 - val_accuracy: 0.8640 - val_loss: 0.3822
Epoch 3/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9576 - loss: 0.1628 - val_accuracy: 0.8520 - val_loss: 0.4603
Epoch 4/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9756 - loss: 0.0884 - val_accuracy: 0.8640 - val_loss: 0.4861


In [21]:
# Evaluate the LSTM on the test set
test_loss, test_accuracy = model.evaluate(test_padded, test_labels)

print(f"LSTM Test Accuracy: {test_accuracy:.2%}")
print(f"LSTM Test Loss: {test_loss:.4f}")

print("\n--- Final Comparison ---")
print(f"Transformer (zero-shot, bart-large-mnli): 70.50% accuracy | 21.7 seconds (inference only, no training)")
print(f"LSTM (trained from scratch on 5000 examples): {test_accuracy:.2%} accuracy")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8000 - loss: 0.5987 
LSTM Test Accuracy: 80.00%
LSTM Test Loss: 0.5987

--- Final Comparison ---
Transformer (zero-shot, bart-large-mnli): 70.50% accuracy | 21.7 seconds (inference only, no training)
LSTM (trained from scratch on 5000 examples): 80.00% accuracy


## Step 3: Attention vs. RNN Memory

An RNN (or LSTM) processes a sequence one element at a time, carrying forward
a single hidden state that summarizes everything seen so far. This means
information from early words has to survive being repeatedly compressed and
overwritten as the sequence gets longer, which is why RNNs struggle with
long-range dependencies. Attention removes this bottleneck: instead of relying
on a single evolving memory, every element can directly look at and weigh the
relevance of every other element in the sequence, regardless of distance. This
direct access preserves long-range context far better, and because there is no
step-by-step dependency, all positions can be processed in parallel, making
Transformers much faster to train than RNNs.

## Comparison Table

| Model | Accuracy | Training Required | Time |
|---|---|---|---|
| Transformer (zero-shot, bart-large-mnli) | 70.50% | None | 30.0 seconds |
| LSTM (trained from scratch, 5000 examples) | 80.00% | Yes (5 epochs, with early stopping) | ~1 minute |

## Step 4: Chosen Architecture

The LSTM will be the project's core model, since it achieved higher accuracy
(80.00% vs. 70.50%) after training on task-specific data. The Transformer's
lower score is expected, as it was used zero-shot with no fine-tuning — a
fine-tuned Transformer would likely outperform the LSTM.